In [ ]:
%reset -f
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import random_split

import joblib
import json
from pathlib import Path

import sys
sys.path.append("/work/home/acychkqsp6/zrm/MasterThesis/tools")
#sys.path.append("../../tools")
import architec as acht
import evaluation as eva
import prepare as prep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import importlib
# importlib.reload(eva)
# importlib.reload(prep)
#importlib.reload(acht)


<module 'architec' from 'd:\\_MyCodes\\MasterThesis\\Principle_design_SynCom\\Fig2\\../../tools\\architec.py'>

In [ ]:
NumSpecies = 20
featgam = NumSpecies**2 - NumSpecies
task_num = 0 # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: oscillation

# The folder storing datasets splited latter.
folder_path = "/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20"
#folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20"

for rep in range(1,16):
    split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_MD_v1/split_MD_{rep}.csv")
    #split_df = pd.read_csv(f"split_MD_v1/split_MD_{rep}.csv")

    #Trainset
    narray_tr = prep.load_split_dataset(
        split_df,
        folder_path,
        #richness_list=[20],
        richness_list=list(range(3,21)),
        subset="tr"
    )

    #Testset
    narray_te = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[20],
        #richness_list=list(range(3,21)),
        subset="te"
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=97000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=20000,
        random_state=42
    )

    train_dataset = acht.CommunityDataset_NS(Xg_tr, XA_tr, Xd_tr, y_tr)
    test_dataset = acht.CommunityDataset_NS(Xg_te, XA_te, Xd_te, y_te)

    n_val = int(len(train_dataset)*0.2)
    n_train = len(train_dataset)-n_val
    train_ds, val_ds = random_split(train_dataset, [n_train,n_val])

    train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

    n_class = NumSpecies
    model = acht.NSTrans2(NumSpecies, n_class).to(device) # Select deep learning architecture.
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    early_stop = acht.EarlyStopping(patience=8, min_delta=1e-4)
    history = {"loss":[], "val_loss":[]}


    best_state = None

    for epoch in range(140):
        model.train()
        train_loss = 0
        
        for Xg,XA,Xd,y in train_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(Xg,XA,Xd)

            loss = criterion(logits,y)
            loss.backward()

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        model.eval()
        val_loss = 0

        with torch.no_grad():

            for Xg,XA,Xd,y in val_loader:

                Xg = Xg.to(device)
                XA = XA.to(device)
                Xd = Xd.to(device)
                y = y.to(device)

                logits = model(Xg,XA,Xd)
                loss = criterion(logits,y)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"train={train_loss:.5f} "
            f"val={val_loss:.5f}"
        )

        if val_loss < early_stop.best_loss:
            best_state = model.state_dict()

        if early_stop.step(val_loss):
            print("Early stopping")
            break

    model.load_state_dict(best_state)

    # Model saving
    save_dir = Path(f"/work/home/acychkqsp6/zrm/MasterThesis/saved_models/num_stable_states_15fold/sp3to20_NS_transpt3_r{rep}")
    #save_dir = Path(f"../../saved_models/num_stable_states_15fold/sp3to20_NS_transpt3_r{rep}")

    save_dir.mkdir(parents=True, exist_ok=True)  # auto-create directory
    torch.save(model.state_dict(), save_dir / "model.pt")
    joblib.dump(scaler_g, save_dir / "scaler_g.pkl")
    joblib.dump(scaler_d, save_dir / "scaler_d.pkl")

    config = {
        "NumSpecies": NumSpecies,
        "featgam": featgam,
        "task_num": task_num,
        "n_class": n_class,
        "embed_dim": 64,
        "num_heads": 4,
        "ff_dim": 128
    }

    with open(save_dir / "config.json", "w") as f:
        json.dump(config, f)

    model.eval()
    preds = []

    with torch.no_grad():

        for Xg, XA, Xd, _ in test_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)

            logits = model(Xg, XA, Xd)
            prob = torch.softmax(logits, dim=1)
            preds.append(prob.cpu().numpy())

    y_prob = np.concatenate(preds)
    y_pred = np.argmax(y_prob, axis=1)
    metrics = eva.eval_multiclass(n_class, y_te, y_prob)
    df_metrics = pd.DataFrame([metrics])
    print(df_metrics.to_string())


tr shape = (77880, 465)
te shape = (19470, 465)
Epoch 1 train=2.09283 val=1.65818
Epoch 2 train=1.61862 val=1.57438
Epoch 3 train=1.58104 val=1.56754
Epoch 4 train=1.56542 val=1.54359
Epoch 5 train=1.55436 val=1.54144
Epoch 6 train=1.54953 val=1.54285
Epoch 7 train=1.54822 val=1.53662
Epoch 8 train=1.54325 val=1.54711
Epoch 9 train=1.54744 val=1.54647
Epoch 10 train=1.54116 val=1.53011
Epoch 11 train=1.53613 val=1.52962
Epoch 12 train=1.53459 val=1.53011
Epoch 13 train=1.53178 val=1.52985
Epoch 14 train=1.53132 val=1.53137
Epoch 15 train=1.52864 val=1.53389
Epoch 16 train=1.52549 val=1.53955
Epoch 17 train=1.52695 val=1.53540
Epoch 18 train=1.52049 val=1.53623
Epoch 19 train=1.52117 val=1.52820
Epoch 20 train=1.51913 val=1.53513
Epoch 21 train=1.51384 val=1.53368
Epoch 22 train=1.51982 val=1.53423
Epoch 23 train=1.51038 val=1.53283
Epoch 24 train=1.51344 val=1.52575
Epoch 25 train=1.50745 val=1.53329
Epoch 26 train=1.50502 val=1.53367
Epoch 27 train=1.50564 val=1.52803
Epoch 28 train=1

KeyboardInterrupt: 